# Channel-wise principal component analysis of dermatological images

## Objective

The red, green, and blue values in dermatological images may be highly correlated. This notebook explores whether channel-wise principal component analysis (PCA) can represent that colour information using fewer input channels.

Each PCA observation is one pixel, and its three features are its red, green, and blue values. This differs from image-wise PCA, where an entire flattened image is one observation.

PCA is fitted exclusively on training-set pixels. Validation and test data remain untouched so they can provide independent assessments later.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA


def find_repository_root(start: Path) -> Path:
    """Find the nearest parent directory containing pyproject.toml."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the repository root")


repository_root = find_repository_root(Path.cwd())
data_directory = repository_root / "data" / "raw_data"
train_images = np.load(data_directory / "train_images.npy")
train_labels = np.load(data_directory / "train_labels.npy").ravel()

print(f"Images: {train_images.shape}; labels: {train_labels.shape}")

## Inspecting the RGB channels

The following figure shows one training image and its red, green, and blue channels. All channel images use the same intensity range so their brightness can be compared directly.

In [ ]:
image_index = 10
example_image = train_images[image_index]
channel_names = ("Red channel", "Green channel", "Blue channel")

fig, axes = plt.subplots(1, 4, figsize=(12, 3), layout="constrained")
axes[0].imshow(example_image)
axes[0].set_title(f"Original image\n(class {train_labels[image_index]})")

for channel_index, channel_name in enumerate(channel_names):
    axes[channel_index + 1].imshow(
        example_image[..., channel_index], cmap="gray", vmin=0, vmax=255
    )
    axes[channel_index + 1].set_title(channel_name)

for axis in axes:
    axis.axis("off")

plt.show()

## Fitting channel-wise PCA

The images are reshaped from (images, height, width, channels) into (pixels, channels). Intensities are converted to floating point and scaled to the range [0, 1]. PCA centres these values before finding three orthogonal linear combinations of RGB.

The component weights show how each component combines red, green, and blue. Component signs are arbitrary: reversing all signs represents the same PCA solution.

In [ ]:
pixel_colours = train_images.reshape(-1, 3).astype(np.float32) / 255.0

pca = PCA(n_components=3)
pca.fit(pixel_colours)

for number, (variance, weights) in enumerate(
    zip(pca.explained_variance_ratio_, pca.components_, strict=True), start=1
):
    weight_text = ", ".join(
        f"{channel}={weight:+.3f}"
        for channel, weight in zip(("R", "G", "B"), weights, strict=True)
    )
    print(f"PC{number}: {variance:.2%} explained variance; {weight_text}")

fig, axis = plt.subplots(figsize=(6, 4), layout="constrained")
axis.bar(("PC1", "PC2", "PC3"), pca.explained_variance_ratio_)
axis.set(
    title="Variance explained by each component",
    xlabel="Principal component",
    ylabel="Proportion of explained variance",
    ylim=(0, 1),
)
plt.show()

## Visualising pixel colours in PCA space

Rendering every training pixel would obscure the distribution, so a reproducible random sample is plotted. Each point is one pixel. Its colour indicates the class of its source image; it is not a label assigned directly to the pixel.

In [ ]:
rng = np.random.default_rng(42)
sample_indices = rng.choice(len(pixel_colours), size=2_000, replace=False)
sample_scores = pca.transform(pixel_colours[sample_indices])

pixels_per_image = np.prod(train_images.shape[1:3])
sample_labels = train_labels[sample_indices // pixels_per_image]

fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
component_pairs = ((0, 1), (1, 2))

for axis, (x_component, y_component) in zip(axes, component_pairs, strict=True):
    scatter = axis.scatter(
        sample_scores[:, x_component],
        sample_scores[:, y_component],
        c=sample_labels,
        cmap="tab10",
        alpha=0.5,
        s=10,
    )
    axis.set_xlabel(
        f"PC{x_component + 1} "
        f"({pca.explained_variance_ratio_[x_component]:.1%} variance)"
    )
    axis.set_ylabel(
        f"PC{y_component + 1} "
        f"({pca.explained_variance_ratio_[y_component]:.1%} variance)"
    )

fig.suptitle("Sampled pixel colours projected into PCA space")
fig.colorbar(scatter, ax=axes, label="Source-image class")
plt.show()

## Reconstructing an image

One image is transformed and reconstructed using increasing numbers of leading components. Missing component scores are set to zero; the inverse transformation then adds the training-pixel mean.

The PC1 score map is a scalar map of the first component value at each pixel. It is not itself an RGB or grayscale reconstruction.

In [ ]:
image_index = 900
original = train_images[image_index].astype(np.float32) / 255.0
pixels = original.reshape(-1, 3)
scores = pca.transform(pixels)

reconstructions = []
for number_of_components in (1, 2, 3):
    retained_scores = np.zeros_like(scores)
    retained_scores[:, :number_of_components] = scores[:, :number_of_components]
    reconstructed_pixels = pca.inverse_transform(retained_scores)
    reconstructed_image = reconstructed_pixels.reshape(original.shape)
    reconstructions.append(np.clip(reconstructed_image, 0.0, 1.0))

pc1_score_map = scores[:, 0].reshape(original.shape[:2])
score_limit = np.abs(pc1_score_map).max()

fig, axes = plt.subplots(1, 5, figsize=(16, 3), layout="constrained")
axes[0].imshow(original)
axes[0].set_title(f"Original image\n(class {train_labels[image_index]})")

score_plot = axes[1].imshow(
    pc1_score_map, cmap="coolwarm", vmin=-score_limit, vmax=score_limit
)
axes[1].set_title("PC1 score map")
fig.colorbar(score_plot, ax=axes[1], label="PC1 score")

for axis, reconstruction, number_of_components in zip(
    axes[2:], reconstructions, (1, 2, 3), strict=True
):
    axis.imshow(reconstruction)
    suffix = "s" if number_of_components > 1 else ""
    axis.set_title(f"{number_of_components} component{suffix}")

for axis in axes:
    axis.axis("off")

plt.show()

## Conclusions and limitations

The explained-variance results and reconstructions show how much RGB pixel variation is retained by one, two, and three channel-wise components. These results describe colour variance; they do not establish which components contain diagnostically useful information.

Before PCA is adopted as preprocessing:

- classification performance should be compared with an RGB baseline;
- validation, test, and inference data must use the PCA fitted on training data rather than fitting PCA again;
- the fitted mean and component weights must be saved as a versioned model artifact;
- the modest saving from reducing only three RGB channels should be weighed against possible information loss;
- validation and test data must remain excluded from exploratory fitting and model-selection decisions until the appropriate evaluation stage.